# Phase 5: Risk-Adjusted Performance-Based Adoption

Goal: replace phase 4's exogenous stochastic diffusion with the certainty-equivalent switching rule from equations (13)-(15). Each non-adopter switches with logistic probability in the score gap `S_t = CE^(A) - CE^(0)`. Compares adoption dynamics and erosion against an exogenous-diffusion baseline at the same parameters and on the same paired shocks.

Reference: section 3.5 of the proposal for equations (13)-(15), section 3.6 for the intra-period timing and the two evaluation channels, and section 4.1 for the demand-adjusted-return R^2 endpoint and the effective-phi diagnostic.

**Why this phase focuses on the demand-adjusted channel.** Phase 4 already established the realised-return self-fulfilling channel (R^2 against `r_{t+1}` rises with adoption, on tight 100-seed bands). This phase is asking a different question: does endogenous performance-based adoption follow the same *underlying* erosion path as exogenous stochastic diffusion, or does it produce a qualitatively different trajectory? The demand-adjusted-return R^2 and the effective AR coefficient are the right diagnostics for that question, because they read the underlying-process effect rather than the self-fulfilling reinforcement. Phase 5 therefore plots both regimes against demand-adjusted R^2 and effective phi, holding all market parameters and seeds fixed. Note that the demand-adjusted return is a counterfactual, not the full regression residual.

In [ ]:
# Number of traders.
N = 200

# Number of periods.
T = 8000

# Market-maker price-impact coefficient (equation 5).
mu = 0.05

# Reduced-form AR(1) coefficient in the return law (equation 6).
phi = 0.25

# Standard deviation of exogenous news shocks.
sigma_news = 0.01

# Standard deviation of individual null orders (equation 8).
sigma_q = 1.0

# Rolling AR window and order for the public forecast.
forecast_window = 250
forecast_p = 1

# Risk-aversion times perceived variance, the denominator in (1) and (3).
risk_scale = 0.001

# Per-trader position cap from equation (3).
q_cap = 0.05

# Trailing window for rolling OOS R^2 and rolling effective phi metrics.
eval_window = 1000

# Hold adoption at zero until rolling metrics first become finite.
adoption_start_t = forecast_window + eval_window

# Stochastic regime: per-period switch probability (matches phase 4 fast).
stochastic_pi = 1e-3

# Performance regime parameters (equations 13-15).
switching_window = 500    # W_A: rolling window for CE computation
switching_a = 1.0         # 'a' in CE = mean - (a/2) var
switching_alpha = -5.0    # baseline term: Lambda(-5) ~ 0.0067 at S=0
switching_beta = 10000.0  # sensitivity: beta * S ~ 1 at S = 1e-4

# Seed shared by both regimes so demand and news shocks are paired.
seed = 71

# Output paths.
fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import simulate
from reflexive_market.metrics import rolling_oos_r2, rolling_phi

In [ ]:
common = dict(
    T=T, N=N, mu=mu, phi=phi,
    sigma_news=sigma_news, sigma_q=sigma_q,
    forecast_window=forecast_window, forecast_p=forecast_p,
    risk_scale=risk_scale, q_cap=q_cap,
    adoption_start_t=adoption_start_t,
)

stochastic = simulate.run(
    rng=np.random.default_rng(seed),
    adoption_pi=stochastic_pi,
    **common,
)
performance = simulate.run(
    rng=np.random.default_rng(seed),
    switching_window=switching_window,
    switching_a=switching_a,
    switching_alpha=switching_alpha,
    switching_beta=switching_beta,
    **common,
)

regimes = [
    {"name": "stochastic diffusion", "colour": "C0", **stochastic},
    {"name": "performance switching", "colour": "C3", **performance},
]

for r in regimes:
    # Demand-adjusted return (counterfactual stripping out contemporaneous
    # price impact). Not the full regression residual.
    demand_adjusted = r["returns"] - mu * r["demand"]
    r["demand_adjusted"] = demand_adjusted
    r["rolling_oos_r2_da"] = rolling_oos_r2(demand_adjusted, r["forecasts"], eval_window)
    r["rolling_phi"] = rolling_phi(r["returns"], eval_window)

In [ ]:
warmup = forecast_window + eval_window
low_lo, low_hi = warmup, warmup + 1500
high_lo, high_hi = T - 1500, T

print(f"input phi:  {phi:+.4f}    risk_scale: {risk_scale:.4f}    q_cap: {q_cap:.4f}")
print()
header = (
    f"{'regime':<24}{'A_low':>8}{'A_high':>9}"
    f"{'R2da_lo':>10}{'R2da_hi':>10}"
    f"{'phi_lo':>9}{'phi_hi':>9}"
)
print(header)
for r in regimes:
    A_low = float(np.nanmean(r["adoption_share"][low_lo:low_hi]))
    A_high = float(np.nanmean(r["adoption_share"][high_lo:high_hi]))
    R2_low = float(np.nanmean(r["rolling_oos_r2_da"][low_lo:low_hi]))
    R2_high = float(np.nanmean(r["rolling_oos_r2_da"][high_lo:high_hi]))
    phi_low = float(np.nanmean(r["rolling_phi"][low_lo:low_hi]))
    phi_high = float(np.nanmean(r["rolling_phi"][high_lo:high_hi]))
    r["summary"] = (A_low, A_high, R2_low, R2_high, phi_low, phi_high)
    print(
        f"{r['name']:<24}{A_low:>8.3f}{A_high:>9.3f}"
        f"{R2_low:>10.4f}{R2_high:>10.4f}"
        f"{phi_low:>9.4f}{phi_high:>9.4f}"
    )

## Adoption trajectories

Stochastic diffusion is exogenous: the rate is fixed at pi each period regardless of how the rule is performing. CE-based switching is endogenous: each non-adopter's switch probability depends on the recent rolling certainty-equivalent gap S_t = CE^(A) - CE^(0). Whenever the advanced rule pays off relative to the null rule (after risk adjustment), more agents switch in.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for r in regimes:
    ax.plot(r["adoption_share"], color=r["colour"], linewidth=1.2, label=r["name"])
ax.set_title("Adoption share: exogenous vs endogenous switching")
ax.set_xlabel("Time period")
ax.set_ylabel("Share of adopters")
ax.set_ylim(-0.02, 1.02)
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_05_adoption_share.png", dpi=150)
plt.show()

## Switching score over time

S_t is defined only for the performance regime. Tracks the certainty-equivalent gap that drives the logistic switch probability. A persistently positive S_t means each non-adopter is being pulled toward the advanced rule; a flip to negative would stall further adoption.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(performance["switching_score"], color="C3", linewidth=0.9)
ax.axhline(0.0, color="k", linewidth=0.5, linestyle="--")
ax.set_title("Performance switching score S_t = CE^(A) - CE^(0)")
ax.set_xlabel("Time period")
ax.set_ylabel("Score")
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_05_switching_score.png", dpi=150)
plt.show()

## Erosion comparison

Plot demand-adjusted R^2 and rolling effective phi against adoption share for both regimes. If the two trajectories collapse onto similar curves the mechanism is purely a function of A; if they diverge the path of adoption matters too. Endogenous switching tends to climb faster once S_t turns favourable, so for the same realised shock stream it visits the high-A regime sooner.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for r in regimes:
    a = r["adoption_share"]
    yr = r["rolling_oos_r2_da"]
    yp = r["rolling_phi"]
    mask_r = np.isfinite(yr)
    mask_p = np.isfinite(yp)
    axes[0].plot(a[mask_r], yr[mask_r], color=r["colour"], linewidth=0.8, alpha=0.7, label=r["name"])
    axes[1].plot(a[mask_p], yp[mask_p], color=r["colour"], linewidth=0.8, alpha=0.7, label=r["name"])
axes[0].axhline(0.0, color="k", linewidth=0.5)
axes[0].set_title("Demand-adjusted OOS R^2 vs adoption share")
axes[0].set_xlabel("Adoption share")
axes[0].set_ylabel("Rolling OOS R^2 (demand-adjusted)")
axes[0].legend()
axes[1].axhline(phi, color="k", linewidth=0.5, linestyle="--", label=f"input phi = {phi}")
axes[1].set_title("Rolling effective phi vs adoption share")
axes[1].set_xlabel("Adoption share")
axes[1].set_ylabel("Rolling lag-1 autocorrelation")
axes[1].legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_05_erosion_vs_adoption.png", dpi=150)
plt.show()

## Save numeric summary

Per-regime headline scalars at the low-adoption and high-adoption windows, plus the run inputs. Full per-period traces reproduce from the seed.

In [ ]:
summary = np.array([r["summary"] for r in regimes])
regime_names = np.array([r["name"] for r in regimes])

# Summary columns:
# 0: A_low  1: A_high  2: R2_da_low  3: R2_da_high
# 4: phi_low  5: phi_high
# Note: R2_da is the demand-adjusted R^2 (counterfactual stripping out the
# self-fulfilment channel), not the full regression residual.
np.savez(
    f"{data_dir}/phase_05_performance_adoption.npz",
    summary=summary,
    regime_names=regime_names,
    phi_input=np.array(phi),
    mu=np.array(mu),
    risk_scale=np.array(risk_scale),
    q_cap=np.array(q_cap),
    forecast_window=np.array(forecast_window),
    eval_window=np.array(eval_window),
    stochastic_pi=np.array(stochastic_pi),
    switching_window=np.array(switching_window),
    switching_a=np.array(switching_a),
    switching_alpha=np.array(switching_alpha),
    switching_beta=np.array(switching_beta),
    T=np.array(T),
    N=np.array(N),
    seed=np.array(seed),
)

## Done when

- Adoption trajectory under endogenous switching is visibly different from the exogenous diffusion baseline (different shape, different timing).
- Switching score S_t evolves with adoption and tracks the recent profitability gap.
- Both regimes display demand-adjusted R^2 erosion and a rising effective phi at high adoption; the two curves overlap at high A but the *path* through A space differs.

What this phase shows: in the absence of transaction costs (phase 7's extension), the advanced rule's certainty-equivalent stays positive even at high adoption (mean adopter profit grows because adopter trades reinforce the AR component in realised returns). So endogenous switching reinforces adoption rather than dampening it. The proposal hypothesises that with transaction costs (or a true performance penalty), the score eventually falls and switching backs off, which is the natural extension to test in phase 7.